# Minimal oxygen regime boundary model

Clean notebook that runs top-to-bottom without errors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
oxygen_species = pd.DataFrame({
    "species": ["O_atom","O2","O2_minus","HO2","H2O2","OH","H2O","O3"],
    "MolecularWeight": [16.00,32.00,32.00,33.00,34.00,17.00,18.00,48.00],
    "electron_affinity": [1.46,0.45,0.45,1.10,1.10,1.83,0.00,2.10],
    "ionization_energy": [13.62,12.07,np.nan,11.00,10.60,13.00,12.62,12.50],
    "formal_charge": [0,0,-1,0,0,0,0,0],
    "n_oxygen": [1,2,2,2,2,1,1,3],
    "has_OO": [0,1,1,1,1,0,0,0]
})

oxygen_species["regime"] = oxygen_species["species"].map({
    "O2": "OO_regime",
    "O2_minus": "OO_regime",
    "HO2": "OO_regime",
    "H2O2": "OO_regime",
    "OH": "OH_regime",
    "H2O": "OH_regime",
    "O_atom": "OH_regime",
    "O3": "other"
})

oxygen_species

In [ ]:
features = [
    "MolecularWeight",
    "electron_affinity",
    "ionization_energy",
    "formal_charge",
    "n_oxygen",
    "has_OO"
]

X_raw = oxygen_species[features].values

imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X_raw)

mean = X_imp.mean(axis=0)
std = X_imp.std(axis=0)
std[std == 0] = 1.0
X_scaled = (X_imp - mean) / std

pca = PCA(n_components=2)
Xp = pca.fit_transform(X_scaled)

oxygen_species["PC1"] = Xp[:, 0]
oxygen_species["PC2"] = Xp[:, 1]

oxygen_species[["species","regime","PC1","PC2"]]

In [ ]:
# Fit a linear boundary on the training regimes only
boundary_df = oxygen_species[oxygen_species["regime"].isin(["OO_regime", "OH_regime"])].copy()
boundary_df["y"] = (boundary_df["regime"] == "OO_regime").astype(int)

clf = LogisticRegression()
clf.fit(boundary_df[["PC1", "PC2"]].values, boundary_df["y"].values)

a, b = clf.coef_[0]
c = clf.intercept_[0]

def signed_distance_to_boundary(x, y, a, b, c):
    return (a*x + b*y + c) / np.sqrt(a*a + b*b)

oxygen_species["boundary_score"] = oxygen_species.apply(
    lambda row: signed_distance_to_boundary(row["PC1"], row["PC2"], a, b, c),
    axis=1
)

oxygen_species[["species","regime","PC1","PC2","boundary_score"]]

In [ ]:
# Visualize the boundary
x_vals = np.linspace(oxygen_species["PC1"].min() - 0.5, oxygen_species["PC1"].max() + 0.5, 300)
if abs(b) > 1e-8:
    y_vals = -(a * x_vals + c) / b
else:
    y_vals = np.full_like(x_vals, 0.0)

plt.figure(figsize=(8, 6))
for regime, group in boundary_df.groupby("regime"):
    plt.scatter(group["PC1"], group["PC2"], s=180, label=regime)
for _, row in boundary_df.iterrows():
    plt.text(row["PC1"] + 0.03, row["PC2"] + 0.03, row["species"], fontsize=11)
plt.plot(x_vals, y_vals, linestyle="--", label="regime boundary")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Boundary between O–O and O–H oxygen regimes")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Boundary score summary
score_df = oxygen_species.sort_values("boundary_score")
score_df[["species","regime","boundary_score"]]

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(score_df["species"], score_df["boundary_score"])
plt.axhline(0, linestyle="--")
plt.ylabel("Signed distance to regime boundary")
plt.title("Oxygen regime boundary score")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# O3 as held-out point
oxygen_species.loc[oxygen_species["species"] == "O3", ["species","PC1","PC2","boundary_score"]]

In [ ]:
# Training accuracy
train_df = oxygen_species[oxygen_species["regime"].isin(["OO_regime", "OH_regime"])].copy()
X_train = train_df[["PC1", "PC2"]].values
y_train = (train_df["regime"] == "OO_regime").astype(int)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_train)
acc = accuracy_score(y_train, y_pred)
print("Training accuracy:", acc)

In [ ]:
# Leave-one-out validation
results = []

for i in range(len(train_df)):
    test = train_df.iloc[i:i+1]
    train = train_df.drop(train_df.index[i])

    X_tr = train[["PC1", "PC2"]].values
    y_tr = (train["regime"] == "OO_regime").astype(int)
    X_te = test[["PC1", "PC2"]].values
    y_te = (test["regime"] == "OO_regime").astype(int)

    loo_model = LogisticRegression()
    loo_model.fit(X_tr, y_tr)
    pred = loo_model.predict(X_te)[0]
    results.append(pred == y_te.values[0])

loo_acc = np.mean(results)
print("LOO accuracy:", loo_acc)
print("Results per point:", results)

In [ ]:
# Predict O3 side
o3 = oxygen_species[oxygen_species["species"] == "O3"]
pred_o3 = model.predict(o3[["PC1", "PC2"]].values)[0]
o3_label = "OO_regime" if pred_o3 == 1 else "OH_regime"
print("O3 predicted as:", o3_label)

In [ ]:
# Bootstrap boundary-angle stability
def compute_boundary_angle(df):
    X = df[features].values
    X = SimpleImputer(strategy="median").fit_transform(X)

    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1.0
    X = (X - mean) / std

    Xp = PCA(n_components=2).fit_transform(X)
    tmp = df.copy()
    tmp["PC1"] = Xp[:, 0]
    tmp["PC2"] = Xp[:, 1]

    oo = tmp[tmp["regime"] == "OO_regime"][["PC1","PC2"]].values
    oh = tmp[tmp["regime"] == "OH_regime"][["PC1","PC2"]].values
    if len(oo) == 0 or len(oh) == 0:
        return np.nan

    v = oo.mean(axis=0) - oh.mean(axis=0)
    if np.linalg.norm(v) == 0:
        return np.nan

    return np.arctan2(v[1], v[0])

angles = []
for _ in range(100):
    sample = oxygen_species.sample(frac=1.0, replace=True)
    angle = compute_boundary_angle(sample)
    if not np.isnan(angle):
        angles.append(angle)

print("Bootstrap mean angle:", np.mean(angles))
print("Bootstrap std angle:", np.std(angles))
print("Valid samples:", len(angles))

In [ ]:
# Correlation of the boundary score with descriptors
corr = oxygen_species[[
    "boundary_score",
    "MolecularWeight",
    "electron_affinity",
    "ionization_energy",
    "formal_charge",
    "n_oxygen",
    "has_OO"
]].corr()

corr["boundary_score"].sort_values()

In [ ]:
# Final summary
print("Training accuracy:", acc)
print("LOO accuracy:", loo_acc)
print("O3 predicted as:", o3_label)
print("Bootstrap mean angle:", np.mean(angles))
print("Bootstrap std angle:", np.std(angles))
print("\nInterpretation: the minimal descriptor system separates O–O and O–H dominated species with local predictive consistency, but the exact boundary orientation is not stable under bootstrap resampling.")